# Homework 2: Machine Learning for Regression

ML Zoomcamp 2026 — reproducible solution using the pinned official dataset.

## Setup and data

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

DATA_URL = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'
COLUMNS = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']
df = pd.read_csv(DATA_URL)[COLUMNS + ['fuel_efficiency_mpg']]

## Q1–Q2. Missing column and median

In [2]:
missing_columns = df.columns[df.isna().any()].tolist()
horsepower_median = df.horsepower.median()
missing_columns, horsepower_median

(['horsepower'], np.float64(254.0))

**Answers:** Q1 `horsepower`; Q2 `254`.

## Model helpers and split

In [3]:
def split_data(data, seed):
    n_val = int(len(data) * 0.2)
    n_test = int(len(data) * 0.2)
    n_train = len(data) - n_val - n_test
    idx = np.arange(len(data))
    np.random.RandomState(seed).shuffle(idx)
    return data.iloc[idx[:n_train]], data.iloc[idx[n_train:n_train+n_val]], data.iloc[idx[n_train+n_val:]]

def train_linear_regression(X, y, r=0):
    X = np.column_stack([np.ones(len(X)), X])
    XTX = X.T @ X
    reg = r * np.eye(XTX.shape[0]); reg[0, 0] = 0
    return np.linalg.solve(XTX + reg, X.T @ y)

def rmse(train, val, fill_value, r=0):
    X_train = train[COLUMNS].fillna(fill_value).to_numpy()
    X_val = val[COLUMNS].fillna(fill_value).to_numpy()
    w = train_linear_regression(X_train, train.fuel_efficiency_mpg.to_numpy(), r)
    pred = np.column_stack([np.ones(len(X_val)), X_val]) @ w
    return mean_squared_error(val.fuel_efficiency_mpg, pred) ** 0.5

train, val, test = split_data(df, 42)

## Q3. Missing-value strategy

In [4]:
scores_q3 = {'With 0': rmse(train, val, 0), 'With mean': rmse(train, val, train.horsepower.mean())}
scores_q3

{'With 0': 2.205293194751563, 'With mean': 2.2018174845685805}

**Answer:** With mean.

## Q4. Regularization

In [5]:
r_values = [0, 0.01, 0.1, 1, 5, 10, 100]
scores_q4 = {r: rmse(train, val, 0, r) for r in r_values}
scores_q4

{0: 2.205293194751563, 0.01: 2.2052931947971612, 0.1: 2.2052931952074486, 1: 2.2052931993112117, 5: 2.2052932175566866, 10: 2.2052932403790595, 100: 2.2052936541437904}

**Answer:** `0`.

## Q5. Stability across seeds

In [6]:
seed_scores = []
for seed in range(10):
    seed_train, seed_val, _ = split_data(df, seed)
    seed_scores.append(rmse(seed_train, seed_val, 0))
std_rmse = np.std(seed_scores)
std_rmse

np.float64(0.0287812740782067)

**Answer:** `0.029`.

## Q6. Final test RMSE

In [7]:
train_9, val_9, test_9 = split_data(df, 9)
combined = pd.concat([train_9, val_9])
test_rmse = rmse(combined, test_9, 0, 0.001)
test_rmse

2.2357747028137993

**Answer:** `2.236`.

## Checks

In [8]:
assert missing_columns[0] == 'horsepower'
assert horsepower_median == 254
assert min(scores_q3, key=scores_q3.get) == 'With mean'
assert min(r_values, key=lambda r: (round(scores_q4[r], 4), r)) == 0
assert round(std_rmse, 3) == 0.029
assert round(test_rmse, 3) == 2.236
print('All Homework 2 checks passed.')

All Homework 2 checks passed.
